# Example Explorer

Pick a MissLearn example, run it, watch it, keep it.

Each of the ten examples is a self-contained study on a real dataset. Choose
one below and run it: the console output and every figure appear here as they
are produced, and a copy of both is written to `explorer_output/` so you can
come back to a run without repeating it.

Some examples download their data on first use and cache it in
`example_data/`. Several take minutes rather than seconds; where a script
offers a reduced setting the **quick** box is available, and the table below
says which those are.


---
## Setup


In [3]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

# Work whether the notebook was opened from examples/ or from the repo root.
_HERE = pathlib.Path.cwd()
if not (_HERE / "example_explorer.py").exists() and \
        (_HERE / "examples" / "example_explorer.py").exists():
    _HERE = _HERE / "examples"
for _p in (str(_HERE), str(_HERE.parent)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

%matplotlib inline

import example_explorer as ex

# Readable on a projector and in a rendered notebook, with padding so a title
# never sits on the axes. Layout engine deliberately not set here: the
# scripts choose their own, and forcing constrained layout on them breaks
# the ones that call tight_layout after drawing a colorbar.
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.titlesize": 16,
    "axes.titlepad": 12,
    "axes.labelpad": 8,
    "figure.dpi": 110,
    "savefig.bbox": "tight",
})

_ = ex.list_examples()


example                       quick  what it shows
---------------------------------------------------------------------------------------------------------------
01_parkinsons_mixed           -      Mixed-effects FIML and multiple imputation on repeated measures
02_thyroid_ensemble           -      Homogeneous and heterogeneous FIML ensembles on native MNAR data
03_wine_pipeline              -      The full workflow: diagnose, validate, fit, explain
04_pima_strategies            -      Six-arm strategy comparison on sentinel-coded clinical data
05_credit_approval_benchmark  -      Finance benchmark: UCI Credit Approval (native missing values)
06_air_quality                yes    Guided MissLearn workflow on atmospheric chemistry data with native holes
07_galaxy_redshift            yes    Photometric redshift from an incomplete multiwavelength catalogue
08_graphene_oxide             yes    Two labels, one missingness pattern: which prediction is more at risk?
09_secom_blockwise        

---
## Choose and run


In [5]:
# =====================================================================
#  Choose an example, then press Run.
#  Output appears below and is written to explorer_output/ when Save is
#  ticked. If ipywidgets is unavailable this falls back to a printed list
#  and you can use the cell further down instead.
# =====================================================================
try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    _HAVE_WIDGETS = True
except ImportError:
    _HAVE_WIDGETS = False

_rows = ex.discover()

if _HAVE_WIDGETS:
    # The dropdown carries the name alone. It clips its own text to the
    # control width whatever layout it is given, so a title placed in the
    # label always lost its ending. The full title of whichever example is
    # selected appears immediately below, and the table above lists them all.
    _choices = ex.picker_labels(_rows)
    pick  = W.Dropdown(options=_choices, value=_rows[0]["name"],
                       description="Example:",
                       layout=W.Layout(width="420px"),
                       style={"description_width": "72px"})
    quick = W.Checkbox(value=True,  description="quick (where supported)")
    save  = W.Checkbox(value=True,  description="save to explorer_output/")
    showf = W.Checkbox(value=True,  description="show figures inline")
    run   = W.Button(description="Run example", button_style="primary",
                     icon="play")
    note  = W.HTML()
    out   = W.Output(layout=W.Layout(border="1px solid #ddd",
                                     padding="6px", max_height="700px",
                                     overflow="auto"))

    def _note(*_):
        r = next(x for x in _rows if x["name"] == pick.value)
        note.value = (
            "<div style='padding:4px 0 2px 0'>"
            "<b style='font-size:1.02em'>%s</b><br>"
            "<span style='color:#666'>%s</span></div>"
            % (r["title"],
               "accepts --quick" if r["quick"] else
               "no --quick option; runs at full size"))
    pick.observe(_note, names="value")
    _note()

    def _run(_b):
        run.disabled = True
        run.description = "Running..."
        with out:
            clear_output(wait=True)
            try:
                ex.run_example(pick.value, quick=quick.value,
                               save=save.value, show=showf.value)
            finally:
                run.disabled = False
                run.description = "Run example"
    run.on_click(_run)

    display(W.VBox([pick, note, W.HBox([quick, save, showf]), run, out]))
else:
    print("ipywidgets is not installed, so the picker is unavailable.")
    print("Use the cell below instead, or: pip install ipywidgets")
    ex.list_examples()


---
## Without widgets

The picker is only a front end for `run_example`, which works on its own.


In [ ]:
# Same thing without widgets. Edit and run.
#   quick=True  uses the reduced setting where a script offers one
#   save=True   writes console.txt and fig01.png, fig02.png, ... under
#               explorer_output/<example>/<timestamp>/
result = ex.run_example("09_secom_blockwise", quick=True, save=True)

# `result` carries where it went and what happened:
#   result.ok, result.seconds, result.n_figures, result.out_dir
#   result.console   the captured text
